In [128]:
import numpy as np 
import matplotlib.pyplot as plt
import OrcFxAPI
import h5py
import pandas as pd
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
from scipy.optimize import minimize


In [129]:
model_path = r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Decay_opzet_harlequin_V3.dat"
sim_path = r"c:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Sim_data\Masscheck_V1"

lookback_window = 5.0      # seconden vóór decay-start waarin we de amplitude zoeken
quiet_window_end = 12.0     # rustige periode eindigt 12 s vóór decay-start
quiet_window_length = 50.0


In [130]:
model = OrcFxAPI.Model(model_path)
constraint = model["decay_constraint"]
floaters = model["floaters"]
floatertype = model["Floatertype"]

In [131]:
constraint.InFrameInitialZ = 2
constraint.InFrameInitialX = 0
constraint.InFrameInitialY = 0
constraint.InFrameInitialAzimuth = 0
constraint.InFrameInitialDeclination = 0 
constraint.InFrameInitialGamma = 0

floatertype.OtherDampingLinearCoeffx = 0 #surge
floatertype.OtherDampingLinearCoeffy = 0 #sway
floatertype.OtherDampingLinearCoeffz  = 0 #heav
floatertype.OtherDampingLinearCoeffRx = 0 #roll
floatertype.OtherDampingLinearCoeffRy = 0 #pitch
floatertype.OtherDampingLinearCoeffRz = 0 #yaw

floatertype.OtherDampingQuadraticCoeffx = 0 #surge
floatertype.OtherDampingQuadraticCoeffy = 0 #sway 
floatertype.OtherDampingQuadraticCoeffz = 0 #heav
floatertype.OtherDampingQuadraticCoeffRx = 0#roll
floatertype.OtherDampingQuadraticCoeffRy = 0 #pitch
floatertype.OtherDampingQuadraticCoeffRz = 0 #yaw


In [132]:
model.RunSimulation()
print("Simulatie klaar.")

Simulatie klaar.


In [133]:
def compute_period(t, z):
    peaks, _ = find_peaks(z)
    
    if len(peaks) < 2:
        return None  # geen betrouwbare periode
    
    T = np.diff(t[peaks])
    return np.mean(T)

In [134]:
target_T = 4.15

mass_values = np.linspace(90,94, 20)  # kies range die logisch is
results = []

for i, mass in enumerate(mass_values):

    model = OrcFxAPI.Model(model_path)
    
    constraint = model["decay_constraint"]
    floaters = model["floaters"]
    floatertype = model["Floatertype"]

    constraint.InFrameInitialZ = 3.8

    floatertype.Mass = mass

    floatertype.OtherDampingLinearCoeffz = 0
    floatertype.OtherDampingQuadraticCoeffz = 0

    model.RunSimulation()

    model.SaveSimulation(f"{sim_path}\{i}_V1.dat")

    print(f"Sim {i} klaar (mass={mass})")

    

<string>:23: SyntaxWarning: invalid escape sequence '\{'
<>:23: SyntaxWarning: invalid escape sequence '\{'
<string>:23: SyntaxWarning: invalid escape sequence '\{'
<>:23: SyntaxWarning: invalid escape sequence '\{'
C:\Users\verav\AppData\Local\Temp\ipykernel_5984\4038068091.py:23: SyntaxWarning: invalid escape sequence '\{'
  model.SaveSimulation(f"{sim_path}\{i}_V1.dat")


Sim 0 klaar (mass=90.0)
Sim 1 klaar (mass=90.21052631578948)
Sim 2 klaar (mass=90.42105263157895)
Sim 3 klaar (mass=90.63157894736842)
Sim 4 klaar (mass=90.84210526315789)
Sim 5 klaar (mass=91.05263157894737)
Sim 6 klaar (mass=91.26315789473684)
Sim 7 klaar (mass=91.47368421052632)
Sim 8 klaar (mass=91.6842105263158)
Sim 9 klaar (mass=91.89473684210526)
Sim 10 klaar (mass=92.10526315789474)
Sim 11 klaar (mass=92.3157894736842)
Sim 12 klaar (mass=92.52631578947368)
Sim 13 klaar (mass=92.73684210526315)
Sim 14 klaar (mass=92.94736842105263)
Sim 15 klaar (mass=93.15789473684211)
Sim 16 klaar (mass=93.36842105263158)
Sim 17 klaar (mass=93.57894736842105)
Sim 18 klaar (mass=93.78947368421052)
Sim 19 klaar (mass=94.0)


In [135]:
results = []

for i, mass in enumerate(mass_values):

    model = OrcFxAPI.Model(f"{sim_path}\{i}_V1.dat")
    floaters = model["floaters"]

    t_sim = model.general.TimeHistory("Time")
    z_sim = floaters.TimeHistory("Z")

    T = compute_period(t_sim, z_sim)

    results.append((mass, T));

<string>:5: SyntaxWarning: invalid escape sequence '\{'
<>:5: SyntaxWarning: invalid escape sequence '\{'
<string>:5: SyntaxWarning: invalid escape sequence '\{'
<>:5: SyntaxWarning: invalid escape sequence '\{'
C:\Users\verav\AppData\Local\Temp\ipykernel_5984\1674216355.py:5: SyntaxWarning: invalid escape sequence '\{'
  model = OrcFxAPI.Model(f"{sim_path}\{i}_V1.dat")


In [136]:
best = min(results, key=lambda x: abs(x[1] - target_T))

print("Beste resultaat:")
print(f"massa = {best[0]:.6f}")
print(f"periode = {best[1]:.6f}")
print(f"afwijking = {abs(best[1] - target_T):.6f}")


Beste resultaat:
massa = 91.684211
periode = 4.146667
afwijking = 0.003333
